# Settimana 16 — API per un agente asincrono

**Obiettivo:** esporre ResearchAgent come API REST con il pattern fire-and-poll:
`POST /research` avvia la ricerca in background e restituisce un `job_id`;
il client fa polling con `GET /jobs/{job_id}` finché lo status non è `completed`.

**Cosa costruisci** in `proyectos/research-agent/`:
- `models.py` — modelli Pydantic per request/response
- `jobs.py` — archivio dei job + macchina a stati
- `api.py` — FastAPI con 6 endpoint
- `test_api.py` — test con TestClient (incluso polling simulato)

**Prerequisiti:** settimana 13 (FastAPI + Docker), settimana 15 (ResearchAgent).

**Costo stimato:** $0.02–0.10 se esegui i test che chiamano l'agente reale.

## Cella 0 — Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "fastapi", "uvicorn[standard]", "httpx", "pytest",
                "python-dotenv"])

import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Definisci OPENAI_API_KEY nel tuo .env"

PROJECT_DIR = Path("../proyectos/research-agent").resolve()
assert PROJECT_DIR.exists(), "Esegui prima il notebook della settimana 15"
print(f"✓ Progetto: {PROJECT_DIR}")

## Cella 1 — Modelli Pydantic (`models.py`)

In [ ]:
MODELS_PY = '''
"""Modelli di request e response per la ResearchAgent API."""
from enum import Enum
from datetime import datetime
from pydantic import BaseModel, Field


class EstadoJob(str, Enum):
    PENDING   = "pending"
    RUNNING   = "running"
    COMPLETED = "completed"
    FAILED    = "failed"


# ── Requests ──────────────────────────────────────────────────────────────────

class ResearchRequest(BaseModel):
    tema: str = Field(
        min_length=10, max_length=300,
        description="Tema da ricercare",
        examples=["Applicazioni dell'IA generativa nell'acquacoltura in Europa"],
    )


# ── Responses ─────────────────────────────────────────────────────────────────

class JobCreatedResponse(BaseModel):
    job_id: str
    status: EstadoJob
    mensaje: str


class JobStatusResponse(BaseModel):
    job_id:        str
    tema:          str
    status:        EstadoJob
    progreso:      str
    creado_en:     datetime
    completado_en: datetime | None = None
    n_fuentes:     int             = 0
    duracion_s:    float           = 0.0
    error:         str             = ""


class ReportResponse(BaseModel):
    job_id:    str
    tema:      str
    informe:   str
    n_fuentes: int
    duracion_s: float


class JobListResponse(BaseModel):
    total: int
    jobs:  list[JobStatusResponse]


class HealthResponse(BaseModel):
    status:      str
    jobs_activos: int


class ErrorResponse(BaseModel):
    detail: str
'''

(PROJECT_DIR / "models.py").write_text(MODELS_PY.strip())
print("✓ models.py generato")

# Verifica che Pydantic validi correttamente
import sys
sys.path.insert(0, str(PROJECT_DIR))

from models import ResearchRequest

# OK
r = ResearchRequest(tema="Intelligenza artificiale nella pesca galiziana")
print(f"✓ Validazione OK: {r.tema}")

# Deve fallire
try:
    ResearchRequest(tema="corto")
    print("✗ Sarebbe dovuto fallire")
except Exception as e:
    print(f"✓ Validazione min_length corretta: {type(e).__name__}")

## Cella 2 — Archivio dei job (`jobs.py`)

Gestisce il ciclo di vita dei job: creare, aggiornare lo stato, consultare, eliminare.

In [ ]:
JOBS_PY = '''
"""Archivio in memoria per il ciclo di vita dei job di ricerca."""
import threading
from dataclasses import dataclass, field
from datetime import datetime
from models import EstadoJob


@dataclass
class Job:
    id:            str
    tema:          str
    status:        EstadoJob      = EstadoJob.PENDING
    progreso:      str            = "In coda"
    informe:       str            = ""
    error:         str            = ""
    creado_en:     datetime       = field(default_factory=datetime.utcnow)
    completado_en: datetime|None  = None
    n_fuentes:     int            = 0
    duracion_s:    float          = 0.0


class JobStore:
    """Archivio thread-safe di job in memoria."""

    def __init__(self):
        self._jobs: dict[str, Job] = {}
        self._lock = threading.Lock()

    def crear(self, job_id: str, tema: str) -> Job:
        job = Job(id=job_id, tema=tema)
        with self._lock:
            self._jobs[job_id] = job
        return job

    def get(self, job_id: str) -> Job | None:
        with self._lock:
            return self._jobs.get(job_id)

    def listar(self) -> list[Job]:
        with self._lock:
            return sorted(self._jobs.values(),
                          key=lambda j: j.creado_en, reverse=True)

    def eliminar(self, job_id: str) -> bool:
        with self._lock:
            return self._jobs.pop(job_id, None) is not None

    def actualizar(self, job_id: str, **kwargs):
        """Aggiorna i campi del job in modo thread-safe."""
        with self._lock:
            job = self._jobs.get(job_id)
            if job:
                for k, v in kwargs.items():
                    setattr(job, k, v)

    def n_activos(self) -> int:
        """Numero di job in stato pending o running."""
        with self._lock:
            return sum(
                1 for j in self._jobs.values()
                if j.status in (EstadoJob.PENDING, EstadoJob.RUNNING)
            )


# Istanza globale — condivisa da tutti gli endpoint
store = JobStore()
'''

(PROJECT_DIR / "jobs.py").write_text(JOBS_PY.strip())
print("✓ jobs.py generato")

# Test unitario dello store
import importlib, jobs
importlib.reload(jobs)
from jobs import JobStore
from models import EstadoJob

s = JobStore()
j = s.crear("abc123", "Tema di prova con caratteri sufficienti")
assert j.status == EstadoJob.PENDING

s.actualizar("abc123", status=EstadoJob.RUNNING, progreso="1/3 sottotask")
assert s.get("abc123").status == EstadoJob.RUNNING
assert s.n_activos() == 1

s.actualizar("abc123", status=EstadoJob.COMPLETED)
assert s.n_activos() == 0

assert s.eliminar("abc123")
assert s.get("abc123") is None

print("✓ JobStore: tutti i test superati")

## Cella 3 — FastAPI con 6 endpoint (`api.py`)

In [ ]:
API_PY = '''
"""ResearchAgent API — pattern fire-and-poll per ricerca asincrona."""
import asyncio
import uuid
import os
from contextlib import asynccontextmanager
from datetime import datetime

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

from models import (
    EstadoJob, ResearchRequest,
    JobCreatedResponse, JobStatusResponse, ReportResponse,
    JobListResponse, HealthResponse,
)
from jobs import store
from agent.agent import ResearchAgent

INFORMES_DIR = os.getenv("INFORMES_DIR", "./informes")


@asynccontextmanager
async def lifespan(app: FastAPI):
    # Avvio
    import pathlib
    pathlib.Path(INFORMES_DIR).mkdir(parents=True, exist_ok=True)
    yield
    # Shutdown: niente da chiudere (lo store vive in memoria)


app = FastAPI(
    title="ResearchAgent API",
    description="Agente di ricerca autonomo — pattern fire-and-poll",
    version="1.0.0",
    lifespan=lifespan,
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


# ── Background task ───────────────────────────────────────────────────────────

async def _ejecutar_investigacion(job_id: str, tema: str):
    """Esegue ResearchAgent in un thread separato per non bloccare l'event loop."""
    store.actualizar(job_id, status=EstadoJob.RUNNING, progreso="Pianificazione...")
    try:
        agente   = ResearchAgent(informes_dir=INFORMES_DIR, verbose=False)
        resultado = await asyncio.to_thread(agente.investigar, tema)

        store.actualizar(
            job_id,
            status        = EstadoJob.COMPLETED,
            progreso      = "Completato",
            informe       = resultado.informe,
            n_fuentes     = resultado.n_fuentes,
            duracion_s    = resultado.duracion_s,
            completado_en = datetime.utcnow(),
        )
    except Exception as exc:
        store.actualizar(
            job_id,
            status   = EstadoJob.FAILED,
            progreso = "Errore",
            error    = str(exc),
        )


# ── Endpoint ──────────────────────────────────────────────────────────────────

@app.get("/health", response_model=HealthResponse)
async def health():
    return HealthResponse(status="ok", jobs_activos=store.n_activos())


@app.post("/research", response_model=JobCreatedResponse, status_code=202)
async def crear_investigacion(req: ResearchRequest) -> JobCreatedResponse:
    """Avvia una ricerca in background. Restituisce job_id immediatamente."""
    job_id = str(uuid.uuid4())[:8]
    store.crear(job_id, req.tema)
    asyncio.create_task(_ejecutar_investigacion(job_id, req.tema))
    return JobCreatedResponse(
        job_id  = job_id,
        status  = EstadoJob.PENDING,
        mensaje = f"Ricerca avviata. Controlla lo stato con GET /jobs/{job_id}",
    )


@app.get("/jobs/{job_id}", response_model=JobStatusResponse)
async def estado_job(job_id: str) -> JobStatusResponse:
    """Restituisce lo stato e il progresso del job."""
    job = store.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail=f"Job {job_id!r} non trovato")
    return JobStatusResponse(
        job_id        = job.id,
        tema          = job.tema,
        status        = job.status,
        progreso      = job.progreso,
        creado_en     = job.creado_en,
        completado_en = job.completado_en,
        n_fuentes     = job.n_fuentes,
        duracion_s    = job.duracion_s,
        error         = job.error,
    )


@app.get("/jobs/{job_id}/report", response_model=ReportResponse)
async def obtener_informe(job_id: str) -> ReportResponse:
    """Restituisce il report completo. Disponibile solo quando status=completed."""
    job = store.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail=f"Job {job_id!r} non trovato")
    if job.status != EstadoJob.COMPLETED:
        raise HTTPException(
            status_code=409,
            detail=f"Il report non è ancora disponibile (status: {job.status})",
        )
    return ReportResponse(
        job_id     = job.id,
        tema       = job.tema,
        informe    = job.informe,
        n_fuentes  = job.n_fuentes,
        duracion_s = job.duracion_s,
    )


@app.get("/jobs", response_model=JobListResponse)
async def listar_jobs() -> JobListResponse:
    """Elenca tutti i job ordinati per data di creazione (più recente per primo)."""
    jobs = store.listar()
    return JobListResponse(
        total=len(jobs),
        jobs=[
            JobStatusResponse(
                job_id=j.id, tema=j.tema, status=j.status,
                progreso=j.progreso, creado_en=j.creado_en,
                completado_en=j.completado_en, n_fuentes=j.n_fuentes,
                duracion_s=j.duracion_s, error=j.error,
            )
            for j in jobs
        ],
    )


@app.delete("/jobs/{job_id}", status_code=204)
async def eliminar_job(job_id: str):
    """Elimina un job dall'archivio."""
    if not store.eliminar(job_id):
        raise HTTPException(status_code=404, detail=f"Job {job_id!r} non trovato")
'''

(PROJECT_DIR / "api.py").write_text(API_PY.strip())
print("✓ api.py generato")
print()
print("Endpoint:")
for linea in API_PY.split("\n"):
    if linea.strip().startswith("@app."):
        print(" ", linea.strip())

## Cella 4 — Test (`test_api.py`)

I test usano `TestClient` di FastAPI. La sfida di questa API è che l'agente impiega 30-120s,
quindi i test veloci **mockano** `ResearchAgent.investigar()` e c'è un test lento
separato che chiama l'agente reale.

In [ ]:
TEST_API_PY = '''
"""
Test per la ResearchAgent API.

Test veloci (mock): pytest test_api.py -v -m "not lento"
Test completi:      pytest test_api.py -v  (richiede OPENAI_API_KEY e ~2min)
"""
import asyncio
import time
import pytest
from unittest.mock import patch, MagicMock
from fastapi.testclient import TestClient

from api import app
from jobs import store
from models import EstadoJob


# ── Fixture: mock dell'agente ─────────────────────────────────────────────────

def _mock_resultado():
    """Risultato simulato di ResearchAgent.investigar()."""
    r = MagicMock()
    r.informe    = "# Report di prova\n\nContenuto simulato."
    r.n_fuentes  = 3
    r.duracion_s = 0.1
    r.ruta_informe = "/tmp/test_informe.md"
    return r


@pytest.fixture
def client():
    """TestClient con mock dell'agente per test veloci."""
    with TestClient(app) as c:
        yield c


# ── Test veloci (mock) ────────────────────────────────────────────────────────

def test_health(client):
    r = client.get("/health")
    assert r.status_code == 200
    assert r.json()["status"] == "ok"


def test_crear_investigacion_devuelve_202(client):
    r = client.post("/research", json={"tema": "Intelligenza artificiale nell'industria agroalimentare galiziana"})
    assert r.status_code == 202
    data = r.json()
    assert "job_id" in data
    assert data["status"] == "pending"


def test_tema_demasiado_corto_da_422(client):
    r = client.post("/research", json={"tema": "corto"})
    assert r.status_code == 422


def test_job_no_existente_da_404(client):
    r = client.get("/jobs/inexistente")
    assert r.status_code == 404


def test_informe_job_pending_da_409(client):
    """Un job in stato pending non ha un report disponibile → 409."""
    r = client.post("/research", json={"tema": "Bioeconomia forestale in Galizia e sue applicazioni"})
    job_id = r.json()["job_id"]

    # Senza aspettare che finisca
    r2 = client.get(f"/jobs/{job_id}/report")
    assert r2.status_code == 409


def test_listar_jobs(client):
    r = client.get("/jobs")
    assert r.status_code == 200
    data = r.json()
    assert "total" in data
    assert "jobs" in data


def test_eliminar_job_existente(client):
    r = client.post("/research", json={"tema": "Tendenze di digitalizzazione nel settore ittico"})
    job_id = r.json()["job_id"]

    r2 = client.delete(f"/jobs/{job_id}")
    assert r2.status_code == 204

    r3 = client.get(f"/jobs/{job_id}")
    assert r3.status_code == 404


def test_eliminar_job_inexistente_da_404(client):
    r = client.delete("/jobs/no-existe")
    assert r.status_code == 404


def test_informe_job_completado(client):
    """Inietta un job completato direttamente nello store e verifica GET /report."""
    from jobs import Job
    from datetime import datetime

    job = Job(
        id="test99",
        tema="Tema di prova sufficientemente lungo",
        status=EstadoJob.COMPLETED,
        informe="# Report\nContenuto di prova.",
        n_fuentes=2,
        duracion_s=5.0,
        completado_en=datetime.utcnow(),
    )
    store._jobs["test99"] = job

    r = client.get("/jobs/test99/report")
    assert r.status_code == 200
    data = r.json()
    assert "informe" in data
    assert data["n_fuentes"] == 2

    # Pulizia
    store.eliminar("test99")


# ── Test lento (agente reale) ─────────────────────────────────────────────────

@pytest.mark.lento
def test_ciclo_completo_con_agente_real(client):
    """
    Test di integrazione completo: crea un job, fa polling fino al completamento
    e verifica che il report abbia contenuto.
    Richiede OPENAI_API_KEY. Può richiedere fino a 2 minuti.
    """
    r = client.post("/research",
        json={"tema": "Applicazioni del machine learning nell'industria conserviera galiziana"})
    assert r.status_code == 202
    job_id = r.json()["job_id"]

    # Polling fino al completamento (max 3 minuti)
    inicio = time.time()
    while time.time() - inicio < 180:
        r2 = client.get(f"/jobs/{job_id}")
        data = r2.json()
        print(f"  Status: {data[\'status\']} | {data[\'progreso\']}")
        if data["status"] == "completed":
            break
        if data["status"] == "failed":
            pytest.fail(f"Job fallito: {data[\'error\']}")
        time.sleep(5)
    else:
        pytest.fail("Timeout: il job non è terminato in 3 minuti")

    # Recupera il report
    r3 = client.get(f"/jobs/{job_id}/report")
    assert r3.status_code == 200
    informe = r3.json()["informe"]
    assert len(informe) > 200, "Il report sembra troppo corto"
    print(f"\nReport ({len(informe)} caratteri):\n{informe[:300]}...")
'''

(PROJECT_DIR / "test_api.py").write_text(TEST_API_PY.strip())
print("✓ test_api.py generato")

## Cella 5 — Eseguire i test veloci

In [ ]:
import subprocess

resultado = subprocess.run(
    [sys.executable, "-m", "pytest", "test_api.py", "-v", "-m", "not lento", "--tb=short"],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
)
print(resultado.stdout)
if resultado.returncode != 0:
    print(resultado.stderr)

## Cella 6 — Client Python per il polling

Un helper riutilizzabile per chiamare l'API da qualsiasi script o notebook.

In [ ]:
CLIENT_PY = '''
"""Client Python per la ResearchAgent API."""
import time
import httpx


class ResearchClient:
    """
    Client che incapsula il ciclo fire-and-poll.

    Uso:
        c = ResearchClient("http://localhost:8000")
        informe = c.investigar_y_esperar("IA nell'acquacoltura", timeout=120)
        print(informe)
    """

    def __init__(self, base_url: str = "http://localhost:8000"):
        self.base = base_url.rstrip("/")
        self.http = httpx.Client(timeout=30)

    def iniciar(self, tema: str) -> str:
        """Avvia una ricerca e restituisce il job_id."""
        r = self.http.post(f"{self.base}/research", json={"tema": tema})
        r.raise_for_status()
        return r.json()["job_id"]

    def estado(self, job_id: str) -> dict:
        r = self.http.get(f"{self.base}/jobs/{job_id}")
        r.raise_for_status()
        return r.json()

    def informe(self, job_id: str) -> str:
        r = self.http.get(f"{self.base}/jobs/{job_id}/report")
        r.raise_for_status()
        return r.json()["informe"]

    def investigar_y_esperar(
        self,
        tema: str,
        timeout: int = 180,
        intervalo: float = 5.0,
        verbose: bool = True,
    ) -> str:
        """
        Avvia una ricerca e fa polling finché non termina.
        Restituisce il report come stringa Markdown.
        """
        job_id = self.iniciar(tema)
        if verbose:
            print(f"Job avviato: {job_id}")

        inicio = time.time()
        while time.time() - inicio < timeout:
            est = self.estado(job_id)
            if verbose:
                elapsed = int(time.time() - inicio)
                print(f"  [{elapsed}s] {est[\'status\']} — {est[\'progreso\']}")

            if est["status"] == "completed":
                return self.informe(job_id)
            if est["status"] == "failed":
                raise RuntimeError(f"Ricerca fallita: {est[\'error\']}")

            time.sleep(intervalo)

        raise TimeoutError(f"Il job {job_id} non è terminato in {timeout}s")
'''

(PROJECT_DIR / "client.py").write_text(CLIENT_PY.strip())
print("✓ client.py generato")
print()
print("Uso del client (con l'API in esecuzione):")
print("""
  from client import ResearchClient

  c = ResearchClient("http://localhost:8000")
  informe = c.investigar_y_esperar("IA nella filiera del vino galiziano")
  print(informe)
""")

## Cella 7 — Avviare l'API in locale (opzionale)

Esegui la cella, poi apri http://localhost:8000/docs nel browser
per vedere la documentazione Swagger e provare gli endpoint manualmente.

**Ferma il kernel per interrompere il server.**

In [ ]:
# Decommenta per avviare il server (blocca il kernel)
# import subprocess, sys
# subprocess.run(
#     [sys.executable, "-m", "uvicorn", "api:app", "--reload", "--port", "8000"],
#     cwd=str(PROJECT_DIR),
# )

print("Per avviare l'API:")
print(f"  cd {PROJECT_DIR}")
print("  uvicorn api:app --reload --port 8000")
print()
print("Docs: http://localhost:8000/docs")
print("Prova rapida:")
print('  curl -X POST http://localhost:8000/research \\')
print('       -H "Content-Type: application/json" \\')
print('       -d \'{"tema": "Applicazioni dell\'IA nella pesca galiziana"}\'\'')

## Sfida — Aggiungere progresso dettagliato durante l'esecuzione

Il campo `progreso` ora dice solo `"Pianificazione..."` o `"Completato"`. Miglioralo affinché
mostri quale sottotask sta eseguendo in tempo reale: `"2/4 sottotask completati"`.

La sfida ha due parti:

**Parte A — Callback in ResearchAgent**

Modifica `ResearchAgent.investigar()` in `agent/agent.py` per accettare un callback
opzionale che viene chiamato dopo ogni sottotask:

```python
def investigar(
    self,
    tema: str,
    on_progreso: Callable[[int, int, str], None] | None = None,
) -> ResultadoInvestigacion:
    ...
    for i, subtarea in enumerate(plan, 1):
        resultado = ejecutar_subtarea(subtarea)
        resultados.append(resultado)
        if on_progreso:
            on_progreso(i, len(plan), subtarea["descripcion"])
```

**Parte B — Aggiornare lo store dal callback**

In `api.py`, passa il callback a `investigar()` affinché aggiorni il campo `progreso`:

```python
def _on_progreso(completadas: int, total: int, desc: str):
    store.actualizar(
        job_id,
        progreso=f"{completadas}/{total} sottotask — {desc[:40]}"
    )

resultado = await asyncio.to_thread(agente.investigar, tema, _on_progreso)
```

Verifica che funzioni facendo polling con `GET /jobs/{job_id}` mentre gira
il test lento e controllando che `progreso` cambi tra una chiamata e l'altra.

## Riepilogo della settimana 16

| Concetto | Dove l'hai applicato |
|---------|-------------------|
| Pattern fire-and-poll | `api.py` + `POST /research` |
| HTTP 202 Accepted | `POST /research` |
| HTTP 409 Conflict | `GET /jobs/{id}/report` quando non completato |
| Macchina a stati (pending→running→completed/failed) | `jobs.py` + `EstadoJob` |
| `asyncio.create_task` per il background | `api.py` |
| `asyncio.to_thread` per non bloccare | `_ejecutar_investigacion()` |
| Test con mock dell'agente | `test_api.py` (8 test veloci) |
| Test di integrazione con agente reale | `test_api.py` (`@pytest.mark.lento`) |
| Client con polling e backoff | `client.py` |
| Progresso in tempo reale | Sfida |

**Il Blocco 7 continua.** La settimana 17 aggiunge LLMOps: osservabilità, trace e metriche di costo
per l'agente in produzione.